In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random
import re
from generate_highd import generate_highd, highd_msg
from generate_expresswayA import generate_expresswayA, expresswayA_msg
from generate_freewayB import generate_freewayB, freewayB_msg
from generate_ngsim import generate_ngsim, ngsim_msg
from generate_ind import generate_ind, ind_msg
from generate_round import generate_round, round_msg
from generate_AV import generate_AV, AV_msg
from langchain.chat_models import ChatOpenAI
from langchain.schema import AIMessage, HumanMessage, SystemMessage

In [2]:
def create_samples(scenario="highd", frame_step=5, num_examples=5, save=True, risk_level=None):
    """
    Randomly selects JSON files from a scenario folder and generates sample descriptions.

    Parameters:
    - scenario (str): One of ['highd', 'freewayB', 'expresswayA', 'ngsim', 'ind', 'round', 'extreme'].
    - frame_step (int): Frame step for description generation.
    - num_examples (int): Number of random samples to generate.
    - save (bool): If True, saves the output to disk.
    - risk_level (str): 'both', 'high', or 'normal' (or 'ttc_1s'/'ttc_2s' for extreme)
    """
    base_folder = r"C:\Users\Administrator\Desktop\KHHRED\event_annotations"
    output_dir = r"C:\Users\Administrator\Desktop\KHHRED\llm_text\samples"
    os.makedirs(output_dir, exist_ok=True)

    # === Default generator function map for non-extreme scenarios ===
    default_generator_map = {
        "highd": generate_highd,
        "freewayB": generate_freewayB,
        "expresswayA": generate_expresswayA,
        "ngsim": generate_ngsim,
        "ind": generate_ind,
        "round": generate_round,
    }

    generator_fn = default_generator_map.get(scenario)

    if scenario in ["ind", "round", "expresswayA", "freewayB", "highd", "ngsim"]:
        base_scenario_folder = os.path.join(base_folder, scenario)
        input_folder_high = os.path.join(base_scenario_folder, f"{scenario}_high_risk")
        input_folder_normal = os.path.join(base_scenario_folder, f"{scenario}_normal_risk")

        if os.path.exists(input_folder_high) and os.path.exists(input_folder_normal):
            if risk_level == "high":
                all_files = [(os.path.join(input_folder_high, f), "high risk") for f in os.listdir(input_folder_high) if f.endswith(".json")]
            elif risk_level == "normal":
                all_files = [(os.path.join(input_folder_normal, f), "normal risk") for f in os.listdir(input_folder_normal) if f.endswith(".json")]
            elif risk_level == "both":
                all_files_high = [(os.path.join(input_folder_high, f), "high risk") for f in os.listdir(input_folder_high) if f.endswith(".json")]
                all_files_normal = [(os.path.join(input_folder_normal, f), "normal risk") for f in os.listdir(input_folder_normal) if f.endswith(".json")]
                all_files = all_files_high + all_files_normal
            else:
                raise ValueError("Invalid risk_level. Choose from ['both', 'high', 'normal'].")
        else:
            # fallback if no split
            input_folder = os.path.join(base_folder, scenario)
            all_files = [(os.path.join(input_folder, f), "normal risk") for f in os.listdir(input_folder) if f.endswith(".json")]

    elif scenario == "extreme":
        base_scenario_folder = os.path.join(base_folder, scenario)
        if risk_level == "ttc_1s":
            input_folder = os.path.join(base_scenario_folder, "ttc_1s")
        elif risk_level == "ttc_2s":
            input_folder = os.path.join(base_scenario_folder, "ttc_2s")
        else:
            raise ValueError("For extreme, risk_level must be 'ttc_1s' or 'ttc_2s'.")

        all_files = [(os.path.join(input_folder, f), risk_level) for f in os.listdir(input_folder) if f.endswith(".json")]

    else:
        raise ValueError(f"Unhandled scenario: {scenario}")

    if len(all_files) == 0:
        raise ValueError(f"No JSON files found under {scenario} with risk level '{risk_level}'.")

    selected_files = random.sample(all_files, min(num_examples, len(all_files)))

    for file_path, risk_label in selected_files:
        filename = os.path.basename(file_path)

        # Extract frame numbers from filename
        match = re.search(r'frame_(\d+)_to_(\d+)', filename)
        if not match:
            raise ValueError(f"❌ Cannot extract frame range from filename: {filename}")

        start_frame = int(match.group(1))
        last_frame = int(match.group(2))
        end_frame = last_frame - 30  # crop off last 30 frames

        # If extreme: dynamically select the correct generator_fn by file prefix
        if scenario == "extreme":
            if filename.lower().startswith("expresswaya"):
                generator_fn = generate_expresswayA
            elif filename.lower().startswith("freewayb"):
                generator_fn = generate_freewayB
            else:
                raise ValueError(f"❌ Unknown file type in 'extreme': {filename}")

        # Generate the description
        description = generator_fn(
            json_path=file_path,
            frame_step=frame_step,
            start_frame=start_frame,
            end_frame=end_frame
        )

        # Add risk label header
        description = f"=== Scenario Risk Classification: {risk_label.upper()} ===\n\n" + description

        if save:
            base_name = filename.replace(".json", "")
            out_path = os.path.join(output_dir, f"{scenario}_desc_{base_name}.txt")
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(description)
            print(f"✅ Saved: {out_path}")

        print("=" * 100)
        print(description[:500] + "\n...")
        print("=" * 100)


In [5]:
'''For Sample Generation'''
# create_samples("expresswayA", risk_level='high')
# create_samples("freewayB")
# create_samples("highd",risk_level='high')
# create_samples("ngsim")
# create_samples("ind")
# create_samples("round")
# create_samples(scenario="extreme",num_examples=300, risk_level='ttc_1s')

'For Samples Generation'

In [ ]:
'''LLM Responses to Different Datasets'''
# llm_response(
#     input_file=r"c:\Users\Administrator\Desktop\KHHRED\llm_text\samples\highd_desc_highd_01_7_followingId_yaw_left.txt",
#     system_message=highd_msg)
# llm_response(
#     input_file=r"c:\Users\Administrator\Desktop\KHHRED\llm_text\samples\expresswayA_track_1_car_812_frame_5670_to_5760_frames_5670_to_5735.txt",
#     system_message=SYSTEM_MESSAGE)
# llm_response(
#     input_file=r"c:\Users\Administrator\Desktop\KHHRED\llm_text\samples\extreme_desc_freewayB_track_3_car_10862_frame_1344_to_1489.txt",
#     system_message=freewayB_msg)

In [ ]:
import os
import re
import time
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage
import openai
from tenacity import retry, wait_exponential, stop_after_attempt

# Function to extract the last frame's text
def extract_last_frame_text(content):
    frames = list(re.finditer(r"Frame (\d+):", content))
    if not frames:
        raise ValueError("No frame found in content.")
    last_frame = frames[-1]
    start_idx = last_frame.start()
    return content[start_idx:].strip(), int(last_frame.group(1))

# Optimized llm_response function with exponential backoff retry
@retry(wait=wait_exponential(min=2, max=60), stop=stop_after_attempt(5))
def llm_response(input_file: str, system_message: str, llm, output_folder: str):
    filename = os.path.basename(input_file)
    output_path = os.path.join(output_folder, f"response_{filename}")

    with open(input_file, "r", encoding="utf-8") as f:
        content = f.read().strip()

    if not content:
        print(f"⚠️ Empty file skipped: {input_file}")
        return

    last_frame_text, last_frame_number = extract_last_frame_text(content)
    print(f"✅ Processing file: {filename}, Last Frame: {last_frame_number}")

    messages = [
        SystemMessage(content=system_message),
        HumanMessage(content=last_frame_text)
    ]

    response = llm(messages).content

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(response)

    print(f"✅ Response saved: {output_path}")

# Main batch-processing loop
def batch_generate_llm_responses(input_folder, output_folder, system_message):
    api_key = "" 
    os.environ["OPENAI_API_KEY"] = api_key

    llm = ChatOpenAI(
        temperature=0.0,
        model_name="gpt-4o",
        max_tokens=2500,
        request_timeout=120,
        streaming=False,
    )

    os.makedirs(output_folder, exist_ok=True)

    txt_files = [f for f in os.listdir(input_folder) if f.endswith('.txt')]
    txt_files.sort()

    print(f"🔁 Found {len(txt_files)} files.")

    for i, filename in enumerate(txt_files, 1):
        input_path = os.path.join(input_folder, filename)
        print(f"\n🚀 [{i}/{len(txt_files)}] {filename}")

        try:
            llm_response(input_path, system_message, llm, output_folder)
        except openai.OpenAIError as e:
            print(f"❌ OpenAI error on {filename}: {e}")
        except Exception as e:
            print(f"❌ Unexpected error on {filename}: {e}")

        time.sleep(1) 


SYSTEM_MESSAGE = """

You are ChatGPT, a large language model trained by OpenAI. You are now acting as a mature driving assistant. And your task is to make the decision that assures the safety.

You have access to the same information as a real human driver, which includes the following:
1. Description of the environment
2. Your own vehicle's position, dynamic information, and legally permissible actions in that lane.
3. Vehicles in your current lane and adjacent lanes, including their position, dynamic information, and legally permissible actions in that lane.

This scenario takes place at an expressway. The positive x-coordinate points east, while the positive y-coordinate points south. The top-left corner is the origin (0,0). The velocity components follow the same convention: a negative vx indicates movement toward the west, while a negative vy indicates movement toward the north.

Your available actions include:
1. IDLE: Remain in the current lane with the current speed (Action ID: 1)
2. Turn Left: Change to the lane on the left of the current lane (Action ID: 2)
3. Turn Right: Change to the lane on the right of the current lane (Action ID: 3) 
4. Acceleration: Increase vehicle speed (Action ID: 4) 
5. Deceleration: Reduce vehicle speed (Action ID: 5)

Safety is the top priority. Below is the template for your reasoning process.

1. **Imminent Collision Warning:** If an imminent collision warning is triggered, immediately prune all unsafe actions (i.e., any movement towards the direction of collision). Retain only those actions that can avoid the collision.

2. **For each vehicle in the surrounding vehicle list (vehicle_ids):**

   TTC for all preceding vehicles = (preceding vehicle's x_coordinate - ego vehicle's x_coordinate) / (ego vehicle's x_velocity - preceding vehicle's x_velocity)
   TTC for all following vehicles = (ego vehicle's x_coordinate - following vehicle's x_coordinate) / following vehicle's x_velocity - ego vehicle's x_velocity)
   - **Preceding Vehicle (vehicle_id):** Check the Time to Collision (TTC). If TTC < 5 seconds, idling and acceleration are not allowed. If TTC < 7 seconds and the preceding vehicle is decelerating, acceleration is not allowed.
   - **Following Vehicle (vehicle_id):** Check the Time to Collision (TTC). If TTC < 5 seconds or (TTC < 7 seconds and the following vehicle is accelerating), braking is not allowed unless there is an imminent collision ahead.
   - **Left Preceding Vehicle (vehicle_id):** Check the linear TTC. If TTC < 5 seconds, turning left is not allowed. If TTC < 5 seconds and the vehicle is turning right, acceleration and idling are not allowed.
   - **Left Following Vehicle (vehicle_id):** Check the linear TTC. If TTC < 5 seconds, turning left is not allowed. If TTC < 5 seconds and the vehicle is turning right, braking and idling are not allowed.
   - **Right Preceding Vehicle (vehicle_id):** Check the linear TTC. If TTC < 5 seconds, turning right is not allowed. If TTC < 5 seconds and the vehicle is turning left, idling and braking are not allowed.
   - **Right Following Vehicle (vehicle_id):** Check the linear TTC. If TTC < 5 seconds, turning right is not allowed. If TTC < 5 seconds and the vehicle is turning left, idling and acceleration are not allowed.
   - **Left Alongside Vehicle (vehicle_id):** If there is a left alongside vehicle, turning left is not allowed.
   - **Right Alongside Vehicle (vehicle_id):** If there is a right alongside vehicle, turning right is not allowed.

3. **Post-Pruning Actions:** After pruning all unsafe actions based on the above rules (note that no collision warning might result in all actions being available after step 1), list all the safest remaining actions. 

4. Now, acting as an experienced human driver, choose the best action ID that ensures both short-term safety and driving smoothness, give a reasoning process to back up your selection. Your response must end with an action ID in the following format: (e.g. Final Decision: ####3)
"""



In [ ]:
'''Response Generation for All Samples'''
# Example usage:
# input_folder = r"C:\Users\Administrator\Desktop\KHHRED\llm_text\samples"
# output_folder = r"C:\Users\Administrator\Desktop\KHHRED\llm_text\responses"

# batch_generate_llm_responses(input_folder, output_folder, SYSTEM_MESSAGE)